## Homework 4: PixArt-α

Напомним, что диффузионная модель учится восстанавливать данные, к которым добавлен шум. В **DDPM** нейросеть предсказывает этот шум, а при генерации её предсказания позволяют постепенно превратить случайный вектор в изображение. Чем больше изображение, тем дороже обрабатывать его на каждом таком шаге.

Чтобы сократить вычисления, можно сначала сжать изображение автокодировщиком. Тогда диффузионная модель будет восстанавливать компактное представление, а декодер переведёт результат обратно в пиксели. На этой идее основаны [Latent Diffusion Models](https://arxiv.org/abs/2112.10752). Задача предсказания шума сохраняется, но решается уже в пространстве **непрерывных латентов**.

Компактную карту латентов удобно разбить на патчи и обрабатывать как последовательность токенов. Так устроен [Diffusion Transformer (DiT)](https://arxiv.org/abs/2212.09748).

Для генерации по описанию к токенам изображения нужно добавить текстовое условие. В [PixArt-α](https://arxiv.org/abs/2310.00426) для этого используется cross-attention.

В этом задании мы соберём небольшую модель PixArt-α и обучим её генерировать пейзажи по тексту. Готовый VAE будет кодировать изображения, готовый T5 — подписи, а трансформер будет предсказывать шум с учётом обоих представлений.


### Задание

Вам предстоит реализовать основные части модели, обучить её на датасете пейзажей `Landscapes HQ` и исследовать, как текст и параметры генерации влияют на результат.

За домашнее задание можно получить **10 баллов**. Заполните пропуски `...` в заготовках кода и ответьте на вопросы. Можно добавлять ячейки и менять внутреннюю организацию решения.

В оценку каждого раздела входят код, результаты его работы и ответы на вопросы.


In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from diffusers import AutoencoderKL, DDPMScheduler, DPMSolverMultistepScheduler
from transformers import AutoTokenizer, T5EncoderModel, CLIPModel, CLIPProcessor
from torchmetrics.image.fid import FrechetInceptionDistance
from tqdm.auto import tqdm


In [ ]:
ROOT = Path(".").resolve()

SEED = 42
GPU_ID = 0
device = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.set_device(device)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
print("Working directory:", ROOT, "| device:", device)


### Задание 1: Подготовка данных (0.5 балла)

Скачайте [изображения LHQ256](https://disk.yandex.ru/d/HPEEntpLv8homg) и [CSV с описаниями](https://drive.google.com/file/d/1LevxpFR5fywVySHzvlze7EYGLgErZLSL/view?usp=sharing).

Подробнее об изображениях можно прочитать в [репозитории LHQ](https://github.com/universome/alis/blob/master/lhq.md).

Подписи получены автоматически по обработанным изображениям. Повторите ту же обработку, чтобы модель видела тот же фрагмент, для которого составлено описание:

- переведём изображение в RGB;
- изменим размер короткой стороны до 144 пикселей с бикубической интерполяцией;
- вырежем по центру квадрат 128×128;
- преобразуем изображение в тензор и нормализуем значения пикселей в [-1, 1].

Нужно:

- скачать и распаковать изображения, скачать CSV с подписями;
- прочитать CSV и сопоставить записи с файлами изображений;
- реализовать описанную обработку, используя бикубическую интерполяцию в `Resize`;
- показать не менее 8 пар из обучающей выборки «картинка — подпись».

В заготовке ниже `records` — список словарей с полями из CSV, а `train_indices` и `test_indices` — индексы обучающих и тестовых записей. При желании можно выбрать другую организацию данных.


In [ ]:
IMAGE_ROOT = ROOT / "data/lhq256"
ARCHIVE_PATH = ROOT / "data/lhq_256.zip"
CSV_PATH = ROOT / "data/captions_lhq.csv"


In [ ]:
# Your code here:
records = ...
train_indices = ...
test_indices = ...


In [ ]:
# Use the same image geometry that was used for captioning.
IMAGE_SIZE = 128
RESIZE_SIZE = 144


In [ ]:
image_transform = transforms.Compose([
    # Your code here: Resize, CenterCrop, ToTensor, Normalize
    ...,
])


In [ ]:
# Your code here: load images and display at least 8 image-caption pairs.
...


**Вопрос.** Сравните подписи с изображениями. Какие заметные детали не попали в текст, а какие были описаны неверно?

**Ваш ответ:**


### Задание 2: VAE (0.5 балла)

Используйте предобученный [VAE](https://huggingface.co/stabilityai/sd-vae-ft-mse). Для изображения 3×128×128 он возвращает параметры распределения латента формы 4×16×16.

Распределение латента имеет вид:

$$q(z\mid x)=\mathcal N\left(\mu(x),\operatorname{diag}\left(\exp(\log\sigma^2(x))\right)\right).$$

При каждом обращении к изображению сэмплируйте новый латент и умножайте его на коэффициент $s$. Масштабирование приводит дисперсию латентов примерно к 1, чтобы их масштаб соответствовал добавляемому шуму:

$$z_0=s\left(\mu(x)+\exp\left(\tfrac12\log\sigma^2(x)\right)\odot\eta\right),
\qquad \eta\sim\mathcal N(0,I),$$

где $s$ берём из `vae.config.scaling_factor`. 

Перед подачей в декодер масштабирование нужно обратить: 
$$\hat x=D(z_0/s)$$

Проверьте реконструкции до обучения генератора. Используйте среднее $\mu(x)$ вместо случайного латента, чтобы результат не зависел от сэмплирования. Потерянные при кодировании детали декодер не сможет восстановить и при генерации.

Реализуйте получение среднего и логарифма дисперсии и декодирование масштабированного латента. Покажите оригиналы и реконструкции для 6 изображений, выведите формы тензоров и число параметров VAE. Диапазон [-1, 1] относится к пикселям; ограничивать им латенты не нужно.

В `diffusers` распределение `vae.encode(images).latent_dist` уже ограничивает `logvar` диапазоном [-30, 20] для численной устойчивости. Используйте его поля `mean` и `logvar`: повторно ограничивать значения не требуется. `logvar` означает логарифм дисперсии. Это ограничение задано в [реализации распределения](https://github.com/huggingface/diffusers/blob/v0.30.3/src/diffusers/models/autoencoders/vae.py#L704-L711).


In [ ]:
VAE_ID = "stabilityai/sd-vae-ft-mse"


In [ ]:
vae = AutoencoderKL.from_pretrained(
    VAE_ID,
    use_safetensors=True,
).to(device).eval()
vae.requires_grad_(False)

latent_scale = float(vae.config.scaling_factor)
LATENT_CHANNELS = int(vae.config.latent_channels)
LATENT_SIZE = IMAGE_SIZE // 8

print(f"VAE parameters: {sum(p.numel() for p in vae.parameters()):,}")
print(f"Latent shape: ({LATENT_CHANNELS}, {LATENT_SIZE}, {LATENT_SIZE})")
print(f"Scaling factor: {latent_scale}")


In [ ]:
@torch.inference_mode()
def encode_image_statistics(images):
    # Return UNSCALED posterior mean and logvar.
    posterior = ...
    return ..., ...

@torch.inference_mode()
def decode_latents(latents):
    # Undo latent scaling and map decoder output to [0,1].
    images = ...
    return ...


In [ ]:
# Your code here: compare 6 original images with their VAE reconstructions.
...


**Вопрос.** Какие детали изменились при реконструкции? Почему эти изменения важно учитывать при оценке генератора? Что произойдёт, если перед декодированием забыть обратить масштабирование латента?

**Ваш ответ:**


### Задание 3: Dataset (1.5 балла)

Для кодирования подписей используйте замороженный `google/t5-v1_1-base`. Он возвращает последовательность векторов размерности 768.

Приведите подписи к длине 120 токенов: 
- Короткие дополните токенами заполнения, длинные обрежьте. 
- Сохраните маску, чтобы cross-attention не учитывал заполнение. 
- Векторы токенов не усредняйте: вниманию нужны представления отдельных токенов.

Результаты VAE и T5 сохраните на диск. При обучении трансформера их можно читать из кеша, не запуская энкодеры повторно.

Вам нужно:

1. Реализовать `encode_texts`: токенизацию с дополнением и обрезанием до заданной длины, получение `last_hidden_state` и булевой маски внимания. Проверить формы выходов на двух предложениях разной длины.
2. Обработать изображения и подписи небольшими батчами и сохранить результаты в `CACHE_DIR`. Для изображений нужны средние и логарифмы дисперсий латентов без масштабирования и сэмплирования; для подписей — последовательности векторных представлений и маски. Предусмотреть загрузку готового кеша при повторном запуске.
3. Реализовать `LandscapeDataset(indices)`, который выбирает записи по переданным индексам и возвращает словарь тензоров: `mean` и `logvar` формы `[4, 16, 16]`, `text` формы `[120, 768]`, `text_mask` формы `[120]` с типом `torch.bool`.
4. Получить один батч через `DataLoader` и проверить формы тензоров.

После подготовки кеша перенесите VAE и T5 на CPU. Когда понадобятся новые подписи, верните T5 на `device`; для декодирования генераций так же верните VAE. Входы и модель должны находиться на одном устройстве. Представления запросов для сравнения CFG можно получить заранее.

Текстовые представления всего датасета занимают много памяти. Можно записывать их в формате `.npy` с помощью `np.lib.format.open_memmap`, а читать через `np.load(..., mmap_mode="r")`. Для вещественных массивов достаточно `float16`. При преобразовании отдельной записи в тензор используйте копию среза массива. Способ хранения и организацию подготовки кеша выбирайте самостоятельно.


In [ ]:
TEXT_MODEL_ID = "google/t5-v1_1-base"
TEXT_MAX_LENGTH = 120


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_ID)
text_encoder = T5EncoderModel.from_pretrained(TEXT_MODEL_ID).to(device).eval()
text_encoder.requires_grad_(False)
TEXT_DIM = text_encoder.config.d_model

In [ ]:
@torch.inference_mode()
def encode_texts(captions):
    tokens = ...
    context = ...
    mask = ...
    return context, mask


In [ ]:
# Your code here: check text representations and masks on two different-length captions.
...


In [ ]:
CACHE_DIR = ROOT / "cache"  # Rebuild the cache if the data or encoders change.
IMAGE_CACHE_BATCH = 16
TEXT_CACHE_BATCH = 32


In [ ]:
# Your code here: prepare and load the disk cache.
...


In [ ]:
class LandscapeDataset(Dataset):
    def __init__(self, indices):
        # Your code here
        ...

    def __len__(self):
        ...

    def __getitem__(self, index):
        # Return mean, logvar, text and text_mask.
        ...


In [ ]:
# Your code here: inspect one DataLoader batch and move both encoders to CPU.
...


### Задание 4: Mini PixArt и adaLN-single (3 балла)

Реализуйте трансформер, предсказывающий шум по зашумлённому латенту $z_t$, времени $t$ и представлению подписи. На схеме показана архитектура PixArt-α, на которую мы будем опираться.

<img src="images/pixart_architecture.png" alt="Архитектура PixArt-α" width="360" style="max-width: 100%; height: auto;">


#### Timestep and Position Embeddings (0.5 балла)

Чтобы сеть различала уровни шума, время $t$ кодируется синусоидальными признаками.

Для $m=\lfloor D/2\rfloor$ зададим частоты и объединим косинусы и синусы:

$$\omega_k=M^{-k/m},\qquad k=0,\ldots,m-1,\qquad M=10000,$$

$$\operatorname{emb}(t)=\left[\cos(t\omega_0),\ldots,\cos(t\omega_{m-1}),\sin(t\omega_0),\ldots,\sin(t\omega_{m-1})\right].$$

Если размерность $D$ нечётная, дополним результат нулём. Для положения патчей аналогично кодируем координаты по двум осям и объединяем их представления.

Реализуйте:

- `sinusoidal_embedding`: преобразование времён формы `[B]` в тензор `[B, D]` по формулам выше; параметр `max_period` задаёт $M$;
- `spatial_embedding`: кодирование каждой координаты в $D/2$ признаков и объединение двух осей в тензор `[1, G*G, D]`, где $G$ — размер сетки патчей.

Размерность $D$ должна делиться на 4, а порядок координат — совпадать с порядком токенов изображения.


In [ ]:
def sinusoidal_embedding(timesteps, dimension, max_period=10_000):
    # Return timestep features with shape [B, D].
    ...

def spatial_embedding(grid_size, dimension):
    # Return 2D position features with shape [1, G*G, D].
    ...


#### PixArtAttention (0.5 балла)

В Self-attention каждый токен изображения получает информацию от других токенов того же изображения. Запросы, ключи и значения вычисляются из одной последовательности.

В cross-attention запросы вычисляются из токенов изображения, а ключи и значения из токенов текста. На выходе остаётся столько же токенов, сколько было в изображении; длина подписи может быть другой.

Реализуйте модуль `PixArtAttention` с несколькими головами внимания и выходной линейной проекцией:

- при `cross=False` вычисляйте запросы, ключи и значения из `x`;
- при `cross=True` вычисляйте запросы из `x`, а ключи и значения — из `context`, а токены заполнения исключайте с помощью `mask`.

Входы и выход модуля:

- `x` — токены изображения формы `[B, N, D]`;
- `context` — токены текста формы `[B, L, D]`;
- `mask` — булева маска текста формы `[B, L]`, где `True` обозначает значимый токен;
- выход — обновлённые токены изображения формы `[B, N, D]`.

Здесь `B` — размер батча, `N` и `L` — длины последовательностей изображения и текста, `D` — размерность признаков. Для self-attention аргументы `context` и `mask` не используются.

In [ ]:
class PixArtAttention(nn.Module):
    def __init__(self, dimension, heads, cross=False):
        super().__init__()
        # Your code here
        ...

    def forward(self, x, context=None, mask=None):
        # Return an updated sequence with the same shape as x.
        ...


#### PixArtBlock (1 балл)

Временное условие задаёт уровень шума. Преобразуйте синусоидальные признаки времени полносвязной сетью в вектор $e(t)\in\mathbb R^D$. Общая для всех блоков сеть вычисляет по нему шесть векторов модуляции:

$$u(t)=\operatorname{Linear}_{D\to6D}(\operatorname{SiLU}(e(t))).$$

Каждый блок прибавляет к этим векторам свою обучаемую таблицу $S_\ell\in\mathbb R^{6\times D}$. Полученные строки задают сдвиг, масштаб и вес остаточной ветви отдельно для Self-attention и полносвязной сети. В этой схеме, **adaLN-single**, сеть обработки времени общая, а таблицы принадлежат отдельным блокам.

Сдвиг $\beta$ и масштаб $\gamma$ применяются к нормализованным признакам:

$$\operatorname{modulate}(h,\beta,\gamma)=h\odot(1+\gamma)+\beta.$$

Блок последовательно применяет Self-attention, cross-attention и полносвязную сеть. Каждая ветвь обновляет токены через остаточное соединение:

$$h\leftarrow h+g_a\odot\operatorname{SelfAttn}(\operatorname{modulate}(\operatorname{LN}_1(h),\beta_a,\gamma_a)),$$
$$h\leftarrow h+\operatorname{CrossAttn}(Q=h,K=c,V=c),$$
$$h\leftarrow h+g_m\odot\operatorname{MLP}(\operatorname{modulate}(\operatorname{LN}_2(h),\beta_m,\gamma_m)).$$

Коэффициенты $g_a$ и $g_m$ задают вклад соответствующих остаточных ветвей. В cross-attention дополнительного временного коэффициента и дополнительной нормализации нет. Его выходную проекцию инициализируем нулями.

В полносвязной ветви используйте слои $D\to4D\to D$ с `nn.GELU(approximate="tanh")` между ними. Коэффициент расширения равен 4.

Реализуйте `PixArtBlock`: создайте необходимые слои и обучаемую таблицу, затем последовательно примените три ветви по формулам выше. На вход `shared_time` поступают шесть общих векторов модуляции; собственная таблица блока прибавляется к ним перед использованием.


In [ ]:
class PixArtBlock(nn.Module):
    def __init__(self, dimension, heads):
        super().__init__()
        # Your code here
        ...

    def forward(self, x, context, mask, shared_time):
        ...


#### MiniPixArt (1 балл)

Трансформер работает с последовательностью, поэтому разобьём карту латентов на патчи 2×2 и спроецируем каждый патч в вектор размерности $D$. Из карты 4×16×16 получится 64 токена. Добавим к ним двумерные позиционные представления, чтобы сохранить информацию о расположении патчей.

Представления текста спроецируйте в размерность $D$ двухслойной полносвязной сетью с GELU. Полученная последовательность $c$ задаёт ключи и значения для cross-attention.

После последнего блока применим финальную нормализацию. Её сдвиг и масштаб получаются из отдельной таблицы 2×D и вектора $e(t)$. Выходная проекция преобразует токены в патчи предсказанного шума, а `unpatchify` соберёт из них карту той же формы, что и входной латент. Выходную проекцию модели также инициализируем нулями.

Для генерации без подписи добавьте в `MiniPixArt` буфер `null_text` формы `[text_length, text_dimension]`, по умолчанию `[120, 768]`. Инициализируйте его как `torch.randn(text_length, text_dimension) / text_dimension**0.5` и зарегистрируйте через `register_buffer`. Буфер сохраняется с весами и переносится вместе с моделью, но не обучается.

Пропускайте `null_text` через ту же проекцию, что и обычную подпись. Для него задайте маску из `True`. В функции потерь обращайтесь к `denoiser.null_text`.

Соберите `MiniPixArt`: преобразование латентов и текста, обработку времени, последовательность блоков и финальный слой (**0.75 балла**). Реализуйте `unpatchify`, который собирает патчи шума обратно в карту формы входного латента (**0.25 балла**).

Внутреннюю организацию классов можно менять.


In [ ]:
class MiniPixArt(nn.Module):
    def __init__(self, latent_size, latent_channels=4, patch_size=2, dimension=384,
                 depth=12, heads=6, text_dimension=768, text_length=120):
        super().__init__()
        self.register_buffer(
            "null_text",
            torch.randn(text_length, text_dimension) / text_dimension**0.5,
        )
        # Your code here
        ...

    def unpatchify(self, tokens):
        ...

    def forward(self, z_t, timesteps, text, text_mask):
        ...


#### Model Initialization

Начните с 12 блоков ширины 384 и 6 голов внимания. Размер сети можно менять под доступный бюджет. Проверьте, что ширина делится на число голов и на 4, а сторона карты латентов — на размер патча.


In [ ]:
# Recommended starting model: approximately 30 million trainable parameters.
MODEL_DIM, MODEL_DEPTH, MODEL_HEADS, PATCH_SIZE = 384, 12, 6, 2


In [ ]:
# Your code here: create the model and report its parameter count.
...


#### Model Checks

До обучения проверьте форму выхода и отсутствие нечисловых значений. Из-за нулевой инициализации выходной проекции предсказание в начале равно нулю, поэтому по нему нельзя проверить влияние текста. При первом обратном проходе ненулевой градиент получает только выходная проекция; нулевые градиенты предыдущих слоёв в этот момент ожидаемы.

- Маску проверьте отдельно на `PixArtAttention` со случайно инициализированными весами: измените только скрытые маской текстовые токены и убедитесь, что выход не изменился. Проверяйте модуль в режиме `eval`.
- После нескольких обновлений весов повторите обратный проход с функцией потерь и проверьте, что градиенты конечны и доходят до более ранних слоёв, включая cross-attention. Для этой проверки не нужно менять предусмотренную инициализацию модели.


In [ ]:
# Your code here: check initial output shape and finite values.
# Test masking on a standalone PixArtAttention with random weights.
# Check gradients in earlier layers after a few training updates.
...


**Вопросы.** Как в модели разделены роли времени и текста? Что даёт общая полносвязная сеть для временного условия при наличии отдельных таблиц в блоках? Как токены заполнения без маски повлияли бы на cross-attention?

**Ваш ответ:**


### Задание 5: Обучение (2 балла)

Для обучения подготовьте зашумлённые латенты и соответствующий им целевой шум.

Для каждого изображения сначала сэмплируем масштабированный латент $z_0$ по сохранённым средним и логарифмам дисперсий. Затем независимо выбираем время $t$ и гауссов шум $\epsilon$, который добавим к этому латенту:

$$z_t=\sqrt{\bar\alpha_t}\,z_0+\sqrt{1-\bar\alpha_t}\,\epsilon.$$

Сеть получает $z_t$, $t$ и текстовое условие $c$, а её предсказание сравнивается с добавленным шумом:

$$\mathcal L=\mathbb E\left\|\epsilon-\epsilon_\theta(z_t,t,c)\right\|^2.$$

Шум для сэмплирования латента VAE и шум для зашумления нужно брать независимо. Первый определяет чистый латент $z_0$, второй служит целью в функции потерь.

Используйте линейное расписание коэффициентов шума на 1000 шагов и `prediction_type="epsilon"`. Для каждого изображения в батче независимо выбирайте время от 0 до 999. Расписание можно взять из готового планировщика.

#### Обучение без текстового условия

При генерации мы будем использовать **classifier-free guidance, CFG**. Для этого одной сети нужно уметь предсказывать шум как с подписью, так и без неё. Поэтому с вероятностью `TEXT_DROPOUT` заменим подпись целиком на `denoiser.null_text` — фиксированный буфер, добавленный в архитектуру в задании 4. Последовательность из буфера проходит через ту же обучаемую проекцию текста. Вместе с представлениями заменяем маску: все токены этой последовательности считаем значимыми.

Реализуйте:

- сэмплирование из апостериорного распределения и масштабирование латента (**0.5 балла**);
- замену текста и маски для обучения без условия (**0.5 балла**);
- выбор времени, добавление шума и MSE в `diffusion_loss` (**1 балл**).

После реализации функции потерь подготовьте цикл обучения. Сохраните кривую функции потерь, промежуточные веса и примеры генерации. Выбор оптимизатора и бюджета обучения остаётся за вами.


In [ ]:
TEXT_DROPOUT = 0.1


In [ ]:
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000, beta_schedule="linear",
    prediction_type="epsilon", clip_sample=False,
)


In [ ]:
def sample_posterior(mean, logvar, scale):
    ...

def replace_text_condition(text, mask, null_text, null_attention, drop):
    ...

def diffusion_loss(denoiser, batch):
    # Your code here
    ...


### Задание 6: Сэмплирование и усиление текстового условия (1 балл)

Во время обучения мы добавляли к чистому латенту шум и учили сеть находить его. При генерации начнём с гауссова шума и будем использовать предсказания сети, чтобы постепенно получить латент изображения. Переход между уровнями шума задаёт **сэмплер**.

Возьмём готовый **DPM-Solver++(2M)**. Это численный метод второго порядка, который использует историю предыдущих предсказаний. Он работает с тем же расписанием шума, что и при обучении, но позволяет выбрать меньшее число шагов генерации. Начальная настройка — 20 шагов; такой бюджет сэмплирования используется и в [PixArt-α](https://arxiv.org/html/2310.00426v3#A5).

На каждом шаге оценим шум дважды: с текстом и без текстового условия, на котором сеть тоже обучалась. Затем объединим предсказания:

$$\hat\epsilon=\epsilon_\theta(z_t,t,\varnothing)
+s\left(\epsilon_\theta(z_t,t,c)-\epsilon_\theta(z_t,t,\varnothing)\right).$$

Это и есть **усиление текстового условия (CFG)**. Разность двух предсказаний показывает, как текст меняет оценку шума, а коэффициент $s$ регулирует вклад этого изменения. При $s=0$ получаем генерацию без условия, при $s=1$ — обычную условную генерацию, при $s>1$ усиливаем влияние текста.

Реализуйте генерацию латентов с DPM-Solver++ и CFG, затем декодируйте их в изображения. Можно использовать `DPMSolverMultistepScheduler` из `diffusers`. Настройки планировщика должны соответствовать расписанию обучения и предсказанию шума. Предсказания с текстом и без него можно получать одним удвоенным батчем или двумя вызовами сети. Для сравнения результатов предусмотрите фиксацию начального шума.


In [ ]:
SAMPLE_STEPS = 20


In [ ]:
@torch.inference_mode()
def sample_latents(denoiser, text, text_mask, steps=None, guidance=3.0, seed=42):
    steps = SAMPLE_STEPS if steps is None else steps
    # Your code here: DPM-Solver++ sampling with classifier-free guidance.
    ...


In [ ]:
# Your code here: decode sampled latents and display images with their prompts.
...


### Обучение модели

Соберите цикл обучения с записью функции потерь и сохранением состояния, из которого можно продолжить работу.

Сначала выполните короткий прогон и проверьте генерацию. Для первого запуска подойдут AdamW, скорость обучения $2\cdot10^{-4}$ и размер батча 64. Затем выберите настройки и число обновлений для основного обучения. Укажите их в решении.

При желании добавьте **EMA** — плавное усреднение весов для генерации. Для ускорения вычислений на A100 можно использовать `torch.autocast` с `torch.bfloat16`.

Покажите кривую функции потерь и генерации на фиксированных запросах на промежуточном и итоговом этапах обучения.

После первых нескольких обновлений выполните проверку градиентов из `Model Checks`: начальные нулевые проекции к этому моменту должны начать обучаться.


In [ ]:
OUTPUT_ROOT = ROOT / "results/lhq_pixart_hw4"
BATCH_SIZE = 64
LEARNING_RATE = 2e-4

# Choose the remaining training settings.


In [ ]:
train_dataset = LandscapeDataset(train_indices)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:
# Your code here: train the model, log the loss and save checkpoints.
...


In [ ]:
# Your code here: show intermediate and final generations on fixed prompts.
...


### Задание 7: CFG и итоговые метрики (1.5 балла)

Оцените влияние CFG на генерацию и вычислите FID и CLIPScore.

**1. Влияние CFG — 0.5 балла.** Выберите не менее 4 текстовых запросов и сравните генерации при 3 значениях CFG, включая 1. Для каждого запроса используйте одинаковый начальный шум и число шагов сэмплирования, чтобы различия определялись силой условия. Покажите результаты в виде сетки и обсудите, как CFG влияет на соответствие тексту, насыщенность и артефакты. Используйте собственные запросы или подписи из обучающей выборки.

Перед генерацией верните VAE на `device`. Если запросы не закодированы заранее, на время их обработки верните туда и T5.

**2. FID и CLIPScore — 1 балл.** Вычислите итоговые метрики на тестовой выборке по протоколу ниже. Укажите число изображений и параметры генерации, объясните полученные значения. Сравните значения с тем, что видно на изображениях. Оценка зависит от корректности расчёта и объяснения результатов, а не от достижения заданного порога.


In [ ]:
# Your code here: compare CFG values on fixed prompts and initial noise.


**Ваши выводы:**

Как CFG меняет соответствие тексту, насыщенность и артефакты? Какое значение вы выбрали для итоговой генерации и почему?


### Итоговая оценка: FID и CLIPScore

Выберите подписи из тестовой выборки и сгенерируйте по одному изображению для каждой. FID оценивает сходство распределений реальных и сгенерированных изображений, а CLIPScore — соответствие генераций подписям.

**FID ↓** сравнивает средние и ковариации признаков сети Inception реальных и сгенерированных изображений. Используйте признаки размерности 2048. 

Для **CLIPScore ↑** вычислите для каждой пары $100\cdot\max(\cos(v_{\text{изобр.}},v_{\text{текст}}),0)$, где $v_{\text{изобр.}}$ и $v_{\text{текст}}$ — векторные представления изображения и подписи, и усредните значения по выборке. Для CLIPScore загрузите `CLIPModel` и `CLIPProcessor` с идентификатором `openai/clip-vit-large-patch14` (CLIP ViT-L/14); тексты длиннее 77 токенов обрезаются.

По умолчанию возьмём 5000 генераций и столько же реальных изображений из тестовой выборки с обработкой 128×128. При ограниченном бюджете допустимо N≥1000, но на малой выборке FID менее устойчив. При сравнении запусков используйте одну и ту же выборку из $N$ изображений. Настройки подбирайте на собственных запросах или обучающих подписях, до расчёта тестовых метрик.

Реализуйте расчёт метрик; можно использовать готовые библиотеки, например `torchmetrics`, и предобученную модель CLIP. Сохраните значения метрик и укажите объём выборки, число шагов генерации и коэффициент CFG. Объясните, согласуются ли результаты с визуальными наблюдениями.

Для FID используйте `FrechetInceptionDistance` из `torchmetrics.image.fid`; для стандартных признаков Inception ему нужен пакет `torch-fidelity`. Они включены в ячейку установки в начале ноутбука.


In [ ]:
# Your code here: select test captions and generate images for evaluation.
...


In [ ]:
# Your code here: compute FID and CLIPScore.
...


**Итоговый вывод.** Какие пейзажи получаются лучше, а какие хуже? Приведите примеры ошибок и объясните, что о качестве модели говорят FID и CLIPScore.

**Ваш ответ:**


In [ ]:
# Leave feedback, suggestions, and impressions about the homework here :)
